# 墙体分割模型训练

**对应论文**：Section 2.3 — FPN + ResNet + Affinity Field Loss

**训练流程**：
```
数据加载（WallSegDataset）
  → FPN+ResNet 模型
  → BCE Loss + Affinity Field Loss（Kendall 不确定性加权）
  → AdamW + Cosine LR
  → 验证集 IoU 评估
  → 保存最优模型
```

## 0. 环境配置

In [1]:
import os, sys, gc, random, logging, time
from pathlib import Path
from dataclasses import dataclass, field
from typing import List, Tuple, Optional, Dict

import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import GradScaler, autocast
from torchvision import transforms
from torchvision.models import resnet50, ResNet50_Weights
from torchvision.ops import FeaturePyramidNetwork
from numpy import genfromtxt
import matplotlib.pyplot as plt

# ── 路径配置 ──
CUBICASA_ROOT = r'E:\JOB\CubiCasa5k'   # ← 改成你的路径
DATA_FOLDER   = r'C:/Users/kawayi_yaling/.cache/kagglehub/datasets/qmarva/cubicasa5k/versions/4/cubicasa5k/cubicasa5k/'
CHECKPOINT_DIR = './checkpoints_wall'

os.chdir(CUBICASA_ROOT)
sys.path.insert(0, CUBICASA_ROOT)
from floortrans.loaders.house import House

logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(message)s')
logger = logging.getLogger(__name__)
%matplotlib inline

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'✓ 环境配置完成')
print(f'  device : {DEVICE}')
if torch.cuda.is_available():
    print(f'  GPU    : {torch.cuda.get_device_name(0)}')
    print(f'  显存   : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

✓ 环境配置完成
  device : cuda
  GPU    : NVIDIA GeForce RTX 4050 Laptop GPU
  显存   : 6.4 GB


## 1. 训练配置

In [4]:
@dataclass
class TrainConfig:
    # 数据
    data_folder:   str   = DATA_FOLDER
    train_file:    str   = 'train.txt'
    val_file:      str   = 'val.txt'
    wall_class_id: int   = 2
    crop_padding:  int   = 10
    min_crop_size: int   = 64

    # 滑动窗口
    tile_size:    int = 512
    tile_overlap: int = 64

    # 增强
    aug_scale_range:      Tuple[float,float] = (0.5, 2.0)
    aug_rotation_degrees: List[int] = field(default_factory=lambda: [0,90,180,270])
    aug_use_flip: bool = True

    # 归一化
    norm_mean: Tuple = (0.485, 0.456, 0.406)
    norm_std:  Tuple = (0.229, 0.224, 0.225)

    # 模型
    backbone:         str = 'resnet50'
    fpn_out_channels: int = 256
    num_classes:      int = 2     # background / wall

    # 训练超参数
    batch_size:   int   = 4
    num_workers:  int   = 0       # Windows 建议 0，Linux/Colab 可设 2-4
    max_epochs:   int   = 50
    learning_rate: float = 1e-4
    weight_decay:  float = 1e-4
    warmup_epochs: int   = 3
    use_amp:       bool  = True   # 混合精度，GPU 训练建议开启

    # 损失函数（论文 Section 2.3）
    use_affinity_loss:    bool = True
    affinity_neighborhood: int = 5

    # 检查点
    checkpoint_dir: str = CHECKPOINT_DIR
    save_top_k:     int = 3

    def validate(self):
        assert Path(self.data_folder).exists()
        assert self.tile_overlap < self.tile_size // 2
        assert self.num_classes >= 2
        Path(self.checkpoint_dir).mkdir(parents=True, exist_ok=True)
        logger.info('✓ 配置校验通过')

CFG = TrainConfig()
CFG.validate()
print(f'batch_size={CFG.batch_size}  epochs={CFG.max_epochs}  lr={CFG.learning_rate}')
print(f'use_amp={CFG.use_amp}  use_affinity={CFG.use_affinity_loss}')
print(f'checkpoint_dir={CFG.checkpoint_dir}')

2026-03-15 14:54:40,923 [INFO] ✓ 配置校验通过


batch_size=4  epochs=50  lr=0.0001
use_amp=True  use_affinity=True
checkpoint_dir=./checkpoints_wall


## 2. 数据集

In [7]:
# ── 复用数据 pipeline 里的函数 ──

def load_sample(data_folder, folder):
    folder   = folder.lstrip('/')
    img_path = os.path.join(data_folder, folder, 'F1_scaled.png')
    svg_path = os.path.join(data_folder, folder, 'model.svg')
    img = cv2.imread(img_path)
    if img is None:
        raise FileNotFoundError(f'图像不存在: {img_path}')
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    house = House(svg_path, h, w)
    seg   = house.get_segmentation_tensor()
    return {'image': img, 'seg': seg}


def crop_to_wall_mask(image, seg, wall_class_id=2, padding=10, min_size=64):
    h, w      = image.shape[:2]
    wall_mask = (seg[0] == wall_class_id)
    if wall_mask.sum() == 0:
        return None
    rows = np.where(wall_mask.any(axis=1))[0]
    cols = np.where(wall_mask.any(axis=0))[0]
    y1 = max(rows[0]  - padding, 0)
    y2 = min(rows[-1] + padding, h)
    x1 = max(cols[0]  - padding, 0)
    x2 = min(cols[-1] + padding, w)
    if (y2-y1) < min_size or (x2-x1) < min_size:
        return None
    return {'image': image[y1:y2, x1:x2], 'seg': seg[:, y1:y2, x1:x2]}


def extract_wall_binary_mask(seg, wall_class_id=2):
    return (seg[0] == wall_class_id).astype(np.uint8)


def augment_sample(image, mask, scale_range=(0.5,2.0),
                   rotation_degrees=[0,90,180,270], use_flip=True):
    scale    = random.uniform(*scale_range)
    h, w     = image.shape[:2]
    new_h, new_w = int(h*scale), int(w*scale)
    image    = cv2.resize(image, (new_w, new_h), interpolation=cv2.INTER_LINEAR)
    mask     = cv2.resize(mask,  (new_w, new_h), interpolation=cv2.INTER_NEAREST)
    angle    = random.choice(rotation_degrees)
    if angle != 0:
        h, w = image.shape[:2]
        M    = cv2.getRotationMatrix2D((w/2,h/2), angle, 1.0)
        cos, sin = abs(M[0,0]), abs(M[0,1])
        new_w = int(h*sin + w*cos)
        new_h = int(h*cos + w*sin)
        M[0,2] += (new_w-w)/2
        M[1,2] += (new_h-h)/2
        image = cv2.warpAffine(image, M, (new_w,new_h), flags=cv2.INTER_LINEAR,
                               borderMode=cv2.BORDER_REFLECT_101)
        mask  = cv2.warpAffine(mask,  M, (new_w,new_h), flags=cv2.INTER_NEAREST,
                               borderMode=cv2.BORDER_CONSTANT, borderValue=0)
    if use_flip and random.random() < 0.5:
        image = cv2.flip(image, 1)
        mask  = cv2.flip(mask,  1)
    return image, mask


class WallSegDataset(Dataset):
    """墙体分割数据集（懒加载）"""

    def __init__(self, cfg: TrainConfig, split: str = 'train'):
        self.cfg      = cfg
        self.is_train = (split == 'train')
        data_file     = cfg.train_file if self.is_train else cfg.val_file
        self.folders  = genfromtxt(cfg.data_folder + data_file, dtype='str')
        logger.info('%s Dataset: %d 个样本', split, len(self.folders))

    def __len__(self):
        return len(self.folders)

    def __getitem__(self, idx):
        folder = self.folders[idx]
        try:
            s = load_sample(self.cfg.data_folder, folder)
            c = crop_to_wall_mask(
                s['image'], s['seg'],
                self.cfg.wall_class_id,
                self.cfg.crop_padding,
                self.cfg.min_crop_size,
            )
            if c is None:
                return self._empty()
            mask = extract_wall_binary_mask(c['seg'], self.cfg.wall_class_id)

            # 随机取一个 tile
            img_src, mask_src = c['image'], mask
            h, w   = img_src.shape[:2]
            stride = self.cfg.tile_size - self.cfg.tile_overlap
            ys     = list(range(0, max(h-self.cfg.tile_size+1,1), stride))
            xs     = list(range(0, max(w-self.cfg.tile_size+1,1), stride))
            if not ys or ys[-1]+self.cfg.tile_size < h: ys.append(max(h-self.cfg.tile_size,0))
            if not xs or xs[-1]+self.cfg.tile_size < w: xs.append(max(w-self.cfg.tile_size,0))
            ty = random.choice(list(set(ys)))
            tx = random.choice(list(set(xs)))
            img_tile  = img_src[ty:ty+self.cfg.tile_size, tx:tx+self.cfg.tile_size].copy()
            mask_tile = mask_src[ty:ty+self.cfg.tile_size, tx:tx+self.cfg.tile_size].copy()
            th, tw = img_tile.shape[:2]
            if th < self.cfg.tile_size or tw < self.cfg.tile_size:
                img_tile  = cv2.copyMakeBorder(img_tile,  0, self.cfg.tile_size-th,
                                               0, self.cfg.tile_size-tw, cv2.BORDER_REFLECT_101)
                mask_tile = cv2.copyMakeBorder(mask_tile, 0, self.cfg.tile_size-th,
                                               0, self.cfg.tile_size-tw, cv2.BORDER_CONSTANT, value=0)

            if self.is_train:
                img_tile, mask_tile = augment_sample(
                    img_tile, mask_tile,
                    self.cfg.aug_scale_range,
                    self.cfg.aug_rotation_degrees,
                    self.cfg.aug_use_flip,
                )
                img_tile  = cv2.resize(img_tile,  (self.cfg.tile_size, self.cfg.tile_size), interpolation=cv2.INTER_LINEAR)
                mask_tile = cv2.resize(mask_tile, (self.cfg.tile_size, self.cfg.tile_size), interpolation=cv2.INTER_NEAREST)

            img_t  = transforms.ToTensor()(img_tile)
            img_t  = transforms.Normalize(self.cfg.norm_mean, self.cfg.norm_std)(img_t)
            mask_t = torch.from_numpy(mask_tile).long()

            del s, c
            gc.collect()
            return {'image': img_t, 'mask': mask_t}

        except Exception as e:
            logger.warning('样本 %s 失败: %s', folder, e)
            return self._empty()

    def _empty(self):
        s = self.cfg.tile_size
        return {'image': torch.zeros(3,s,s), 'mask': torch.zeros(s,s).long()}


# ── 验证数据集 ──
train_ds = WallSegDataset(CFG, split='train')
val_ds   = WallSegDataset(CFG, split='val')
print(f'train: {len(train_ds)} 样本')
print(f'val  : {len(val_ds)} 样本')

sample = train_ds[0]
print(f'image shape : {sample["image"].shape}  range=[{sample["image"].min():.2f},{sample["image"].max():.2f}]')
print(f'mask  shape : {sample["mask"].shape}   唯一值={sample["mask"].unique().numpy()}')

2026-03-15 14:54:42,381 [INFO] train Dataset: 4200 个样本
2026-03-15 14:54:42,382 [INFO] val Dataset: 400 个样本


train: 4200 样本
val  : 400 样本
image shape : torch.Size([3, 512, 512])  range=[-2.12,2.64]
mask  shape : torch.Size([512, 512])   唯一值=[0 1]


## 3. 模型定义

**论文 Section 2.3**：FPN + ResNet50

In [9]:
class WallSegModel(nn.Module):
    """
    FPN + ResNet50 墙体二分类分割模型
    输出：[B, 2, H, W] logits
    """
    def __init__(self, num_classes=2, fpn_channels=256):
        super().__init__()
        backbone        = resnet50(weights=ResNet50_Weights.IMAGENET1K_V2)
        self.layer0     = nn.Sequential(backbone.conv1, backbone.bn1,
                                        backbone.relu, backbone.maxpool)
        self.layer1     = backbone.layer1   # stride 4,   256ch
        self.layer2     = backbone.layer2   # stride 8,   512ch
        self.layer3     = backbone.layer3   # stride 16, 1024ch
        self.layer4     = backbone.layer4   # stride 32, 2048ch
        self.fpn        = FeaturePyramidNetwork(
            in_channels_list=[256, 512, 1024, 2048],
            out_channels=fpn_channels,
        )
        self.head = nn.Sequential(
            nn.Conv2d(fpn_channels, 128, 3, padding=1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, num_classes, 1),
        )

    def forward(self, x):
        size = x.shape[-2:]
        x    = self.layer0(x)
        c2   = self.layer1(x)
        c3   = self.layer2(c2)
        c4   = self.layer3(c3)
        c5   = self.layer4(c4)
        fpn  = self.fpn({'0':c2,'1':c3,'2':c4,'3':c5})
        ref  = fpn['0'].shape[-2:]
        fused = fpn['0']
        for k in ['1','2','3']:
            fused = fused + F.interpolate(fpn[k], size=ref,
                                          mode='bilinear', align_corners=False)
        logits = self.head(fused)
        return F.interpolate(logits, size=size, mode='bilinear', align_corners=False)


model = WallSegModel(num_classes=CFG.num_classes,
                     fpn_channels=CFG.fpn_out_channels).to(DEVICE)
params = sum(p.numel() for p in model.parameters()) / 1e6
print(f'模型参数量：{params:.1f}M')

# 验证前向传播
dummy = torch.randn(1, 3, 512, 512).to(DEVICE)
with torch.no_grad():
    out = model(dummy)
print(f'输入 shape：{dummy.shape}')
print(f'输出 shape：{out.shape}  (应为 [1, 2, 512, 512])')

模型参数量：27.1M
输入 shape：torch.Size([1, 3, 512, 512])
输出 shape：torch.Size([1, 2, 512, 512])  (应为 [1, 2, 512, 512])


## 4. 损失函数

**论文 Section 2.3**：BCE + Affinity Field Loss，用 Kendall 不确定性自动加权

In [30]:
class AffinityFieldLoss(nn.Module):
    def __init__(self, neighborhood_size=5):
        super().__init__()
        self.ks  = neighborhood_size
        self.pad = neighborhood_size // 2

    def forward(self, pred, target):
        B, C, H, W = pred.shape
        pred_prob   = torch.softmax(pred, dim=1)
        target_flat = target.unsqueeze(1).float()
        target_nb   = F.unfold(target_flat, kernel_size=self.ks, padding=self.pad)
        target_nb   = target_nb.view(B, self.ks*self.ks, H, W)
        affinity_gt = (target_nb == target_flat).float()
        pred_nb     = F.unfold(pred_prob, kernel_size=self.ks,
                               padding=self.pad).view(B, C, self.ks*self.ks, H, W)
        pred_center = pred_prob.unsqueeze(2).expand_as(pred_nb)
        cos_sim     = F.cosine_similarity(
            pred_nb.reshape(B,C,-1), pred_center.reshape(B,C,-1), dim=1
        ).view(B, self.ks*self.ks, H, W)

        # ← 修复：用 logits 版本，AMP 安全
        # cos_sim 范围 [-1,1]，直接作为 logits 传入
        return F.binary_cross_entropy_with_logits(
            cos_sim, affinity_gt, reduction='mean'
        )


class WallSegLoss(nn.Module):
    def __init__(self, use_affinity=True):
        super().__init__()
        self.use_affinity = use_affinity
        if use_affinity:
            self.affinity_loss = AffinityFieldLoss()
            self.log_var_bce   = nn.Parameter(torch.zeros(1))
            self.log_var_aff   = nn.Parameter(torch.zeros(1))

    def forward(self, logits, target):
        bce = F.cross_entropy(logits, target)
        if not self.use_affinity:
            return {'loss': bce, 'bce': bce}
        aff = self.affinity_loss(logits, target)

        # ← 修复：确保参数和 loss 在同一个设备上
        log_var_bce = self.log_var_bce.to(bce.device)
        log_var_aff = self.log_var_aff.to(aff.device)

        loss = (torch.exp(-log_var_bce)*bce + log_var_bce +
                torch.exp(-log_var_aff)*aff + log_var_aff)
        return {'loss': loss, 'bce': bce, 'affinity': aff}


criterion = WallSegLoss(use_affinity=CFG.use_affinity_loss)
criterion = criterion.to(DEVICE)
print('损失函数初始化完成')
print(f'  use_affinity={CFG.use_affinity_loss}')

# 验证损失计算
dummy_logits = torch.randn(2, 2, 64, 64)
dummy_target = torch.randint(0, 2, (2, 64, 64))
loss_dict    = criterion(dummy_logits, dummy_target)
print(f'  验证 loss={loss_dict["loss"].item():.4f}')

损失函数初始化完成
  use_affinity=True
  验证 loss=1.6644


## 5. 训练工具

In [ ]:
class AverageMeter:
    """滑动平均统计量"""
    def __init__(self): self.reset()
    def reset(self): self.val=self.avg=self.sum=self.count=0.0
    def update(self, val, n=1):
        self.val  = val
        self.sum += val * n
        self.count += n
        self.avg  = self.sum / self.count


class CheckpointManager:
    """保存 top-k 最优模型，自动删除旧的"""
    def __init__(self, save_dir, top_k=3):
        self.save_dir = Path(save_dir)
        self.save_dir.mkdir(parents=True, exist_ok=True)
        self.top_k    = top_k
        self._records = []   # [(score, path)]

    def save(self, model, optimizer, epoch, metrics):
        score = metrics.get('val_iou', 0.0)
        path  = self.save_dir / f'epoch_{epoch:04d}_iou_{score:.4f}.pth'
        torch.save({
            'epoch'              : epoch,
            'model_state_dict'   : model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'metrics'            : metrics,
        }, path)
        logger.info('保存检查点：%s  val_iou=%.4f', path.name, score)
        self._records.append((score, path))
        self._records.sort(key=lambda x: x[0], reverse=True)
        while len(self._records) > self.top_k:
            _, old = self._records.pop()
            if old.exists(): old.unlink()
            logger.info('删除旧检查点：%s', old.name)


def compute_iou(pred, target, num_classes=2):
    """计算 mean IoU"""
    iou_list = []
    for cls in range(num_classes):
        p = (pred == cls)
        t = (target == cls)
        inter = (p & t).sum().float()
        union = (p | t).sum().float()
        if union > 0:
            iou_list.append((inter/union).item())
    return float(np.mean(iou_list)) if iou_list else 0.0


ckpt_manager = CheckpointManager(CFG.checkpoint_dir, top_k=CFG.save_top_k)
print('✓ 训练工具初始化完成')

## 6. 优化器 & 学习率调度

In [ ]:
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

# 模型参数 + 不确定性权重参数一起优化
all_params = list(model.parameters()) + list(criterion.parameters())
optimizer  = AdamW(all_params, lr=CFG.learning_rate, weight_decay=CFG.weight_decay)
scheduler  = CosineAnnealingLR(
    optimizer,
    T_max=CFG.max_epochs - CFG.warmup_epochs,
    eta_min=CFG.learning_rate * 0.01,
)
scaler = torch.amp.GradScaler(device=DEVICE, enabled=CFG.use_amp and DEVICE=='cuda')

print(f'优化器  : AdamW  lr={CFG.learning_rate}  weight_decay={CFG.weight_decay}')
print(f'调度器  : CosineAnnealingLR  T_max={CFG.max_epochs - CFG.warmup_epochs}')
print(f'AMP     : {CFG.use_amp and DEVICE=="cuda"}')

## 7. DataLoader

In [ ]:
train_loader = DataLoader(
    train_ds,
    batch_size=CFG.batch_size,
    shuffle=True,
    num_workers=CFG.num_workers,
    pin_memory=(DEVICE == 'cuda'),
    drop_last=True,
)
val_loader = DataLoader(
    val_ds,
    batch_size=CFG.batch_size,
    shuffle=False,
    num_workers=CFG.num_workers,
    pin_memory=(DEVICE == 'cuda'),
)

print(f'train_loader : {len(train_loader)} batches')
print(f'val_loader   : {len(val_loader)} batches')

# 验证一个 batch
batch = next(iter(train_loader))
print(f'\nbatch image : {batch["image"].shape}')
print(f'batch mask  : {batch["mask"].shape}')

## 8. 训练 & 验证函数

In [ ]:
def train_epoch(model, criterion, optimizer, scaler, loader, device, epoch, warmup_epochs, lr):
    model.train()
    criterion.train()
    meters = {k: AverageMeter() for k in ['loss','bce','affinity']}
    t0     = time.time()

    if epoch <= warmup_epochs:
        lr_scale = epoch / warmup_epochs
        for pg in optimizer.param_groups:
            pg['lr'] = lr * lr_scale

    for batch in loader:
        images = batch['image'].to(device)
        masks  = batch['mask'].to(device)

        optimizer.zero_grad()

        # ← 修复：新版 autocast 写法
        with torch.amp.autocast(device_type=device, enabled=scaler.is_enabled()):
            logits    = model(images)
            loss_dict = criterion(logits, masks)

        scaler.scale(loss_dict['loss']).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        scaler.step(optimizer)
        scaler.update()

        n = images.size(0)
        for k, m in meters.items():
            if k in loss_dict:
                m.update(loss_dict[k].item(), n)

    elapsed = time.time() - t0
    return {k: m.avg for k,m in meters.items()}, elapsed


@torch.no_grad()
def val_epoch(model, criterion, loader, device, num_classes):
    model.eval()
    criterion.eval()
    loss_m = AverageMeter()
    iou_m  = AverageMeter()

    for batch in loader:
        images = batch['image'].to(device)
        masks  = batch['mask'].to(device)
        logits = model(images)
        loss_d = criterion(logits, masks)
        preds  = logits.argmax(dim=1)
        iou    = compute_iou(preds, masks, num_classes)
        n      = images.size(0)
        loss_m.update(loss_d['loss'].item(), n)
        iou_m.update(iou, n)

    return {'val_loss': loss_m.avg, 'val_iou': iou_m.avg}


print('✓ 训练/验证函数定义完成')

## 9. 训练主循环

In [ ]:
train_history = {'loss':[], 'bce':[], 'affinity':[], 'val_loss':[], 'val_iou':[]}
best_iou      = 0.0

print(f'开始训练，共 {CFG.max_epochs} 个 epoch')
print(f'{'Epoch':>6}  {'train_loss':>10}  {'bce':>8}  {'aff':>8}  {'val_iou':>8}  {'lr':>8}  {'time':>6}')
print('-' * 70)

for epoch in range(1, CFG.max_epochs + 1):

    # ── 训练 ──
    train_metrics, elapsed = train_epoch(
        model, criterion, optimizer, scaler,
        train_loader, DEVICE, epoch,
        CFG.warmup_epochs, CFG.learning_rate,
    )

    # ── 验证 ──
    val_metrics = val_epoch(model, criterion, val_loader, DEVICE, CFG.num_classes)

    # ── 学习率更新（warmup 结束后）──
    if epoch > CFG.warmup_epochs:
        scheduler.step()

    current_lr = optimizer.param_groups[0]['lr']

    # ── 记录 ──
    for k, v in train_metrics.items():
        if k in train_history: train_history[k].append(v)
    train_history['val_loss'].append(val_metrics['val_loss'])
    train_history['val_iou'].append(val_metrics['val_iou'])

    # ── 保存检查点 ──
    all_metrics = {**train_metrics, **val_metrics}
    ckpt_manager.save(model, optimizer, epoch, all_metrics)

    # ── 记录最优 ──
    if val_metrics['val_iou'] > best_iou:
        best_iou = val_metrics['val_iou']
        torch.save(model.state_dict(),
                   f'{CFG.checkpoint_dir}/best_model.pth')

    # ── 打印 ──
    aff_str = f"{train_metrics.get('affinity',0):.4f}"
    print(f'{epoch:>6}  {train_metrics["loss"]:>10.4f}  '
          f'{train_metrics["bce"]:>8.4f}  {aff_str:>8}  '
          f'{val_metrics["val_iou"]:>8.4f}  '
          f'{current_lr:>8.6f}  {elapsed:>5.1f}s')

print(f'\n训练完成！最优 val_iou={best_iou:.4f}')
print(f'最优模型保存在：{CFG.checkpoint_dir}/best_model.pth')

## 10. 训练曲线可视化

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

epochs = range(1, len(train_history['loss'])+1)

axes[0].plot(epochs, train_history['loss'],  label='train_loss')
axes[0].plot(epochs, train_history['val_loss'], label='val_loss')
axes[0].set_title('Loss', fontsize=13)
axes[0].set_xlabel('Epoch'); axes[0].legend(); axes[0].grid(True)

axes[1].plot(epochs, train_history['val_iou'], color='green', label='val_iou')
axes[1].axhline(y=best_iou, color='red', linestyle='--', label=f'best={best_iou:.4f}')
axes[1].set_title('Validation IoU', fontsize=13)
axes[1].set_xlabel('Epoch'); axes[1].legend(); axes[1].grid(True)

if train_history['affinity']:
    axes[2].plot(epochs, train_history['bce'],      label='bce')
    axes[2].plot(epochs, train_history['affinity'], label='affinity')
    axes[2].set_title('Loss Components', fontsize=13)
    axes[2].set_xlabel('Epoch'); axes[2].legend(); axes[2].grid(True)

plt.tight_layout()
plt.savefig(f'{CFG.checkpoint_dir}/training_curves.png', dpi=120)
plt.show()
print(f'训练曲线已保存')

## 11. 加载最优模型验证推理

In [ ]:
# 加载最优模型
best_ckpt = f'{CFG.checkpoint_dir}/best_model.pth'
model.load_state_dict(torch.load(best_ckpt, map_location=DEVICE))
model.eval()
print(f'✓ 最优模型已加载：{best_ckpt}')

# 取一个验证样本推理
val_sample = val_ds[0]
img_t  = val_sample['image'].unsqueeze(0).to(DEVICE)
mask_gt = val_sample['mask'].numpy()

with torch.no_grad():
    logits    = model(img_t)
    pred_mask = logits.argmax(dim=1)[0].cpu().numpy()

iou = compute_iou(
    torch.tensor(pred_mask), torch.tensor(mask_gt), CFG.num_classes
)
print(f'单样本 IoU：{iou:.4f}')

# 反归一化显示
mean = torch.tensor(CFG.norm_mean).view(3,1,1)
std  = torch.tensor(CFG.norm_std).view(3,1,1)
img_show = (val_sample['image'] * std + mean).permute(1,2,0).numpy().clip(0,1)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(img_show)
axes[0].set_title('输入图像', fontsize=12); axes[0].axis('off')
axes[1].imshow(mask_gt,  cmap='gray')
axes[1].set_title('Ground Truth', fontsize=12); axes[1].axis('off')
axes[2].imshow(pred_mask, cmap='gray')
axes[2].set_title(f'预测结果  IoU={iou:.4f}', fontsize=12); axes[2].axis('off')
plt.tight_layout(); plt.show()